# File I/O

So far we have discussed how to process data and how
to build, train, and test deep learning models.
However, at some point we will hopefully be happy enough
with the learned models that we will want
to save the results for later use in various contexts
(perhaps even to make predictions in deployment).
Additionally, when running a long training process,
the best practice is to periodically save intermediate results (checkpointing)
to ensure that we do not lose several days' worth of computation
if we trip over the power cord of our server.
Thus it is time to learn how to load and store
both individual weight vectors and entire models.
This section addresses both issues.


In [1]:
import torch
from torch import nn
from torch.nn import functional as F

## (**Loading and Saving Tensors**)

For individual tensors, we can directly
invoke the `load` and `save` functions
to read and write them respectively.
Both functions require that we supply a name,
and `save` requires as input the variable to be saved.


In [2]:
x = torch.arange(4)
torch.save(x, 'x-file')

We can now read the data from the stored file back into memory.


In [3]:
x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

We can [**store a list of tensors and read them back into memory.**]


In [4]:
y = torch.zeros(4)
torch.save([x, y],'x-files')
x2, y2 = torch.load('x-files')
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

We can even [**write and read a dictionary that maps
from strings to tensors.**]
This is convenient when we want
to read or write all the weights in a model.


In [5]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

## [**Loading and Saving Model Parameters**]

Saving individual weight vectors (or other tensors) is useful,
but it gets very tedious if we want to save
(and later load) an entire model.
After all, we might have hundreds of
parameter groups sprinkled throughout.
For this reason the deep learning framework provides built-in functionalities
to load and save entire networks.
An important detail to note is that this
saves model *parameters* and not the entire model.
For example, if we have a 3-layer MLP,
we need to specify the architecture separately.
The reason for this is that the models themselves can contain arbitrary code,
hence they cannot be serialized as naturally.
Thus, in order to reinstate a model, we need
to generate the architecture in code
and then load the parameters from disk.
(**Let's start with our familiar MLP.**)


In [6]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.LazyLinear(256)
        self.output = nn.LazyLinear(10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)

Next, we [**store the parameters of the model as a file**] with the name "mlp.params".


In [7]:
torch.save(net.state_dict(), 'mlp.params')

To recover the model, we instantiate a clone
of the original MLP model.
Instead of randomly initializing the model parameters,
we [**read the parameters stored in the file directly**].


In [8]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): LazyLinear(in_features=0, out_features=256, bias=True)
  (output): LazyLinear(in_features=0, out_features=10, bias=True)
)

Since both instances have the same model parameters,
the computational result of the same input `X` should be the same.
Let's verify this.


In [9]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

## Summary

The `save` and `load` functions can be used to perform file I/O for tensor objects.
We can save and load the entire sets of parameters for a network via a parameter dictionary.
Saving the architecture has to be done in code rather than in parameters.

## Exercises

1. Even if there is no need to deploy trained models to a different device, what are the practical benefits of storing model parameters?
1. Assume that we want to reuse only parts of a network to be incorporated into a network having a different architecture. How would you go about using, say the first two layers from a previous network in a new network?
1. How would you go about saving the network architecture and parameters? What restrictions would you impose on the architecture?


[Discussions](https://discuss.d2l.ai/t/61)


1. Even if there is no need to deploy trained models to a different device, what are the practical benefits of storing model parameters?


* You can load it again and test it whenever you want to 
* You can share it with your friends for them to test it out
        

1. Even if there is no need to deploy trained models to a different device, what are the practical benefits of storing model parameters?

**Answer:**

The ability to save and load model parameters provides several crucial practical benefits even when deployment isn't the goal:

**Checkpointing during training:**
* Training deep learning models can take days or even weeks. Saving parameters periodically creates checkpoints that protect against unexpected failures (power outages, system crashes, etc.)
* If training is interrupted, you can resume from the latest checkpoint rather than starting over - saving enormous amounts of time and computational resources
* This is especially critical for models trained on cloud instances where continued runtime is expensive

**Experimentation and reproducibility:**
* Stored models serve as experimental artifacts that can be referenced later when documenting results
* They enable scientific reproducibility - other researchers can verify your results using your exact model
* You can compare different training approaches by saving models at various stages and evaluating them systematically

**Model selection and ensemble methods:**
* You can save models at different epochs to analyze how performance evolves over training time
* Create ensemble models by combining predictions from multiple saved models trained with different initializations or architectures
* Save the best performing model based on validation metrics rather than just using the final model

**Transfer learning and fine-tuning:**
* Store pre-trained models that can later be fine-tuned for specific tasks
* Extract specific layers from stored models to use as feature extractors in new architectures
* This creates a library of reusable components that accelerate future model development

**Version control and audit trail:**
* Maintain a history of model evolution as you experiment with different architectures and hyperparameters
* Document which version produced which results for reporting and publication
* Create an audit trail that demonstrates how you arrived at your final model

**Collaboration:**
* Share trained models with colleagues for validation and feedback
* Enable team members to build upon each other's work asynchronously
* Facilitate distributed research where different aspects of model development happen in parallel

By storing model parameters, you transform ephemeral computational artifacts into persistent intellectual assets that can be studied, improved, combined, and leveraged long after the initial training process is complete.

2. Assume that we want to reuse only parts of a network to be incorporated into a network having a different architecture. How would you go about using, say the first two layers from a previous network in a new network?


 I will create a module using nn.sequential and use that to interchange different things.

2. Assume that we want to reuse only parts of a network to be incorporated into a network having a different architecture. How would you go about using, say the first two layers from a previous network in a new network?

**Answer:**

Reusing parts of a neural network (transfer learning) is a powerful technique that allows us to leverage knowledge from previously trained models. To reuse the first two layers of a network in a new architecture, we can follow these steps:

**Conceptual approach:**
1. Load the saved parameters of the original network
2. Create a new network architecture that incorporates the layers we want to reuse
3. Selectively copy parameters from the original network to the corresponding layers in the new network
4. Decide whether to freeze or fine-tune the transferred layers

**PyTorch implementation:**

```python
# Step 1: Define and load the original network
original_net = MLP()
original_net.load_state_dict(torch.load('mlp.params'))

# Step 2: Define a new network architecture
class NewNetwork(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        # Extract the first two layers from the original network
        self.transferred_layers = nn.Sequential(
            original_model.hidden,  # First layer from original model
            nn.ReLU()              # Activation function
        )
        
        # Add new layers specific to our new task
        self.new_layer1 = nn.Linear(256, 128)  # Assumes hidden layer output is 256 dim
        self.new_layer2 = nn.Linear(128, 5)    # New output layer for different task
        
    def forward(self, x):
        # Use the transferred layers first
        x = self.transferred_layers(x)
        # Then pass through the new layers
        x = F.relu(self.new_layer1(x))
        return self.new_layer2(x)

# Step 3: Create the new network with transferred layers
new_net = NewNetwork(original_net)

# Step 4: Optionally freeze the transferred layers to prevent updates
# during training of the new network
for param in new_net.transferred_layers.parameters():
    param.requires_grad = False  # Remove this line if you want to fine-tune these layers

Additional considerations:

* Parameter compatibility: Ensure the dimensions match between layers being connected.
* Adaptation layers: Sometimes you may need adapter layers between transferred and new components.
* Layer access: If the original model doesn't expose layers directly, you might need to create a new model that exactly mimics the original architecture, then load parameters and extract specific parts.
* Batch normalization: Be cautious with batch normalization layers which store running statistics - these should be reset or carefully transferred.
* Module hooks: For more complex architectures, you can use PyTorch hooks to extract intermediate activations from specific layers.

This approach provides a flexible framework for reusing network components while integrating them into new architectures, allowing for powerful transfer learning applications.

3. How would you go about saving the network architecture and parameters? What restrictions would you impose on the architecture?

The model network can be saved as a .py file. The parameters as a dict or json. As for restrictions, I can't think of any, Maybe something that cannot change during run time.

Let me provide a detailed and comprehensive answer to this question about saving network architecture and parameters.

```markdown
3. How would you go about saving the network architecture and parameters? What restrictions would you impose on the architecture?

**Answer:**

Saving both network architecture and parameters requires addressing two distinct challenges: preserving the computational graph structure and storing learned weights. Here's a comprehensive approach:

**Saving Complete Models:**

In PyTorch, we can save both architecture and parameters using `torch.save()` with the entire model:

```python
# Save complete model (architecture + parameters)
torch.save(net, 'complete_model.pth')

# Load complete model
loaded_model = torch.load('complete_model.pth')
```

However, this approach has limitations since it pickles the Python objects, which can cause compatibility issues.

**Better Approach: Separating Architecture and Parameters:**

1. **Architecture Preservation:**
   * Define model architecture in code (Python class or configuration file)
   * Use a serializable configuration format (JSON, YAML) to store hyperparameters
   ```python
   # Architecture config as dictionary
   model_config = {
       'hidden_layers': [256, 128],
       'activation': 'relu',
       'output_dim': 10
   }
   
   # Save architecture configuration
   import json
   with open('model_architecture.json', 'w') as f:
       json.dump(model_config, f)
   ```

2. **Parameter Storage:**
   * Save state_dict (parameter dictionary) separately
   ```python
   # Save parameters only
   torch.save(net.state_dict(), 'model_parameters.pth')
   ```

3. **Model Reconstruction:**
   ```python
   # Load architecture config
   with open('model_architecture.json', 'r') as f:
       config = json.load(f)
       
   # Reconstruct model from config
   new_model = MLP(hidden_dims=config['hidden_layers'], 
                  output_dim=config['output_dim'])
   
   # Load parameters
   new_model.load_state_dict(torch.load('model_parameters.pth'))
   ```

**Necessary Restrictions for Architectures:**

1. **Deterministic Construction:**
   * Architecture must be fully reconstructable from saved configuration
   * Avoid random architectural decisions that aren't captured by configuration

2. **Serialization Compatibility:**
   * Avoid custom Python objects that can't be properly serialized
   * Ensure compatibility with the serialization protocol (e.g., pickle in PyTorch)

3. **Version Management:**
   * Include library version information with saved models
   * Handle backwards compatibility for models created with older library versions

4. **Custom Components:**
   * Custom layers, loss functions, or activation functions must be registered or included
   * Document any custom modules needed to reconstruct the architecture

5. **Environment Independence:**
   * Avoid dependencies on global variables or environment-specific settings
   * Ensure reproducibility across different environments

6. **Hardware Agnosticism:**
   * Architecture should be device-agnostic (can move between CPU/GPU)
   * Handle precision differences (e.g., float32 vs float16)

**Modern Tools and Frameworks:**

For production environments, consider standard model serialization formats:
* ONNX (Open Neural Network Exchange)
* TorchScript
* TensorFlow SavedModel

These formats enable platform-independent model preservation and deployment with both architecture and parameters intact.

By implementing these practices and restrictions, you can creatodel saving workflows that ensure reproducibility, portability, and long-term model viability.
```